In [ ]:
%load_ext autoreload
%autoreload 2

import triton
import triton.language as tl
from triton._C.libtriton import ir
from triton.backends.compiler import GPUTarget
from triton.backends.nvidia.compiler import CUDAOptions
from triton.compiler.code_generator import ast_to_ttir
from triton.compiler.compiler import ASTSource
from triton.backends.metal.compiler import MetalBackend
from triton.backends.metal.msl_generator import MSLKernel


@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask=mask)


context = ir.context()
ir.load_dialects(context)

target = GPUTarget("metal", 0, 32)  # dummy
options = CUDAOptions()  # dummy

src = ASTSource(
    fn=add_kernel,
    signature={"x_ptr": "*fp32", "y_ptr": "*fp32", "output_ptr": "*fp32", "n_elements": "i32"},
    constexprs={"BLOCK_SIZE": 1024},
)

mod = src.make_ir(target, options, codegen_fns={}, module_map={}, context=context)
metadata = {}
mod = MetalBackend.make_ttir(mod, metadata, options)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [138]:
kernel = MSLKernel(mod)


In [139]:
msl = kernel.generate_msl_code()

Processed function add_kernel with 4 arguments: ['!tt.ptr<f32>', '!tt.ptr<f32>', '!tt.ptr<f32>', 'i32']
Processing op: arith.constant
Processed op: arith.constant(i32 1024) -> (i32 c1024_i32)
Processing op: tt.get_program_id
Processed op: tt.get_program_id(i32 0) -> (i32 v0)
Processing op: arith.muli
Processed op: arith.muli(i32 v0, i32 c1024_i32) -> (i32 v1)
Processing op: tt.make_range
Processed op: tt_make_range(i32 0, i32 1024) -> (tensor<1024xi32> v2)
Processing op: tt.splat
Processed op: tt_splat(i32 v1) -> (tensor<1024xi32> v3)
Processing op: arith.addi
Processed op: arith.addi(tensor<1024xi32> v3, tensor<1024xi32> v2) -> (tensor<1024xi32> v4)
Processing op: tt.splat
Processed op: tt_splat(i32 arg_3) -> (tensor<1024xi32> v5)
Processing op: arith.cmpi
Processed op: arith.cmpi(tensor<1024xi32> v4, tensor<1024xi32> v5) -> (tensor<1024xi1> v6) if slt
Processing op: tt.splat
Processed op: tt_splat(!tt.ptr<f32> arg_0) -> (tensor<1024x!tt.ptr<f32>> v7)
Processing op: tt.addptr
Processe

In [140]:
print(msl)

kernel void add_kernel(device float * arg_0, device float * arg_1, device float * arg_2, device int & arg_3, uint thread_threadgroup_idx [[thread_position_in_threadgroup]], uint threadgroup_grid_idx [[threadgroup_position_in_grid]])
{
	const int c1024_i32 = 1024;
	int v0 = threadgroup_grid_idx;
	int v1 = v0 * c1024_i32;
	int v2 = thread_threadgroup_idx;
	int v3 = v1;
	int v4 = v3 + v2;
	int v5 = arg_3;
	bool v6 = v4 < v5;
	device float * v7 = arg_0;
	device float * v8 = v7 + v4;
	float v9;
	if (v6) {
	    v9 = *v8;
	}
	device float * v10 = arg_1;
	device float * v11 = v10 + v4;
	float v12;
	if (v6) {
	    v12 = *v11;
	}
	float v13 = v9 + v12;
	device float * v14 = arg_2;
	device float * v15 = v14 + v4;
	if (v6) {
	    *v15 = v13;
	}
	// return from function
}


In [59]:
mod.get_entry_func_name()

'add_kernel'